In [ ]:

import pyrosetta
from benchmark import bpti_ryfyn_benchmark
from rotamer_extraction import extract_top_n_rotamers
from hamiltonian_creation import extract_hamiltonian_tensors, build_ising_hamiltonian, reduce_hamiltonian
from initialisation import initialize_rosetta

import numpy as np
import pennylane as qml

initialize_rosetta(pyrosetta, extra_flags="-mute all")

In [ ]:
benchmark_pose = bpti_ryfyn_benchmark()

results = extract_top_n_rotamers(benchmark_pose)
pyrosetta.rosetta.core.io.pdb.dump_pdb(benchmark_pose, "benchmark_pose.pdb")

In [ ]:
rotamer_lib, ig, rot_sets = results
h_linear, J_quadratic = extract_hamiltonian_tensors(rotamer_lib, ig, rot_sets)


In [ ]:
h_flex_linear, J_flex_quadratic, global_offset = reduce_hamiltonian(h_linear, J_quadratic, rotamer_lib)
for idx in h_linear:
    print(h_linear[idx])
    print(h_flex_linear.get(idx, "None"))
    print("\n---------------------------------------\n")

In [ ]:
hamiltonian = build_ising_hamiltonian(h_flex_linear, J_flex_quadratic, global_offset)

In [ ]:
print(hamiltonian)

In [11]:
import pennylane as qml
from pennylane import numpy as np

H_ising = hamiltonian
# Assuming 'H_ising' is the Hamiltonian returned by your build_ising_hamiltonian function
num_qubits = 19
dev = qml.device('lightning.gpu', wires=range(num_qubits))

# Define the standard X-mixer for QAOA
mixer_h = qml.dot([1.0] * num_qubits, [qml.PauliX(i) for i in range(num_qubits)])
# THE STANDARD MIXER DIDN'T WORK, CUSTOM MIXER

seq_positions = sorted(list(h_flex_linear.keys()))
wire_offsets = {}
current_wire = 0
for seq in seq_positions:
    wire_offsets[seq] = current_wire
    current_wire += len(h_flex_linear[seq])

def build_xy_mixer(wire_offsets, h_flex):
    """
    Constructs the XY mixer Hamiltonian, restricting mixing strictly
    to intra-residue rotamer wires.
    """
    coeffs = []
    observables = []

    seq_positions = sorted(list(h_flex.keys()))
    for seq in seq_positions:
        base_wire = wire_offsets[seq]
        num_rots = len(h_flex[seq])

        # Apply XY mixing between all pairs of rotamers within THIS residue
        for i in range(num_rots):
            for j in range(i + 1, num_rots):
                w_i = base_wire + i
                w_j = base_wire + j

                # 0.5 * (X_i X_j + Y_i Y_j)
                coeffs.extend([0.5, 0.5])
                observables.extend([
                    qml.PauliX(w_i) @ qml.PauliX(w_j),
                    qml.PauliY(w_i) @ qml.PauliY(w_j)
                ])

    return qml.Hamiltonian(coeffs, observables)

# Instantiate the custom mixer
H_mixer_xy = build_xy_mixer(wire_offsets, h_flex_linear)

def qaoa_layer(gamma, beta):
    qml.qaoa.cost_layer(gamma, H_ising)
    qml.qaoa.mixer_layer(beta, mixer_h)

@qml.qnode(dev)
def cost_function(params):
    # params is a 2D array of shape (2, p) where p is the number of QAOA layers
    gammas = params[0]
    betas = params[1]

    # 1. Initialize in a superposition state
    for i in range(num_qubits):
        qml.Hadamard(wires=i)

    # 2. Apply p layers of QAOA
    for i in range(len(gammas)):
        qaoa_layer(gammas[i], betas[i])

    # 3. Measure the expectation value of the cost Hamiltonian
    return qml.expval(H_ising)

# Define depth 'p'. Start small (p=2) on your M1 to gauge execution time before scaling.
p = 8
np.random.seed(42)
# Initialize parameters close to zero to avoid barren plateaus
initial_params = np.random.uniform(low=-0.01, high=0.01, size=(2, p), requires_grad=True)

DeviceError: Device lightning.gpu does not exist. Make sure the required plugin is installed.

In [ ]:
# Re-declare device and QNode for optimization ensuring backprop

@qml.qnode(dev, diff_method="adjoint")
def cost_function(params):
    # Hadamard Transform
    for i in range(num_qubits):
        qml.Hadamard(wires=i)

    # Applying the qaoa layers
    for i in range(len(params[0])):
        qaoa_layer(params[0][i], params[1][i])
    return qml.expval(H_ising)

# Optimization Execution
opt = qml.AdamOptimizer(stepsize=0.05)
epochs = 150
current_params = initial_params
lowest_param_set = (float('inf'), current_params)

print("Commencing QAOA Optimization...")
for epoch in range(epochs):
    current_params, cost = opt.step_and_cost(cost_function, current_params)
    if cost < lowest_param_set[0]:
        lowest_param_set = (cost, current_params)
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | Cost: {cost:.4f}")

print("Optimization converged.")

In [ ]:
@qml.qnode(dev)
def sample_function(params):
    for i in range(num_qubits):
        qml.Hadamard(wires=i)
    for i in range(len(params[0])):
        qaoa_layer(params[0][i], params[1][i])
    return qml.probs(wires=range(num_qubits))

# Generate the probability vector for all 2^19 states
print(f"Lowest parameter cost: {lowest_param_set[0]}")
probabilities = sample_function(lowest_param_set[1])
# probabilities = sample_function(current_params)

In [ ]:
top_k = 100
top_indices = np.argsort(probabilities)[-top_k:][::-1]
top_indices

In [ ]:

# 1. Extract the top N most probable bitstrings
top_k = 100
# np.argsort returns indices; we take the last 'top_k' and reverse them for descending order
top_indices = np.argsort(probabilities)[-top_k:][::-1]

valid_conformations = []

# Helper to convert integer index back to binary list (length = 19)
def int_to_bitstring(idx, length):
    return [int(x) for x in format(idx, f'0{length}b')]

# 2. Enforce the One-Hot Constraint
for idx in top_indices:
    bitstring = int_to_bitstring(idx, num_qubits)
    is_valid = True

    # Iterate through each residue's allocated wires using your existing `wire_offsets`
    # and the known length of h_flex[seq]
    for seq in seq_positions:
        start_wire = wire_offsets[seq]
        num_rots = len(h_flex_linear[seq])

        # Sum the bits corresponding to this residue's rotamers
        residue_sum = sum(bitstring[start_wire : start_wire + num_rots])

        if residue_sum != 1:
            is_valid = False
            break # Fails the penalty constraint

    if is_valid:
        # 3. Calculate True Biological Energy (Classical PyRosetta Equation)
        # using the valid bitstring against the original h_flex and J_flex tensors.
        bio_energy = 0 # calculate_classical_energy(bitstring, h_flex, J_flex, global_offset)
        valid_conformations.append({
            "bitstring": bitstring,
            "probability": probabilities[idx],
            "energy": bio_energy
        })

if not valid_conformations:
    raise ValueError("Zero valid conformations found in the top sampled states. You must increase QAOA depth 'p' or increase the penalty multiplier.")

# Sort the strictly valid conformations by their true biological energy
valid_conformations.sort(key=lambda x: x['energy'])
best_conformation = valid_conformations[0]

print(f"Optimal Valid Sequence: {best_conformation['bitstring']}")
print(f"Classical Energy: {best_conformation['energy']} kcal/mol")